<a href="https://colab.research.google.com/github/victorgarciaalcalde-ctrl/03MIAR-AlgoritmosOptimizacion/blob/main/TRABAJO_PRACTICO/trabajo_practico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Víctor García Alcalde  <br>
Url: https://github.com/victorgarciaalcalde-ctrl/03MIAR-AlgoritmosOptimizacion/tree/main<br>
Google Colab: https://colab.research.google.com/drive/1Qs-uniimQo6G5-1yJMC3S-zOEInuvG5K#scrollTo=hVbXYX-RfPWh <br>
Problema:
>1. Sesiones de doblaje <br>
>2. Organizar los horarios de partidos de una jornada de La Liga<br>
>3. Configuración de Tribunales

Organizar los horarios de partidos de una jornada de La Liga

• Desde la La Liga de fútbol profesional se pretende organizar los horarios de los partidos de liga de cada jornada. Se conocen algunos datos que nos deben llevar a diseñar un algoritmo que realice la asignación de los partidos a los horarios de forma que maximice la audiencia.






                                        

#Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?

#Representación del espacio de soluciones

El espacio de soluciones se representa como un vector de 10 posiciones (una por cada partido de la jornada). Cada posición contiene un índice que corresponde a uno de los 10 horarios disponibles:

$S = [h_0, h_1, ..., h_{9}]$ donde $h_i \in \{0, 1, ..., 9\}$.

donde cada valor representa un horario (V20, S12, S16, S18, S20, D12, D16, D18, D20, L20).

#Función objetivo

El objetivo es maximizar la audiencia total de la jornada:

$$f(x) = \sum_{i=1}^{10} (\text{Base}_i \times \text{Ponderación}_{h} \times (1 - \text{Reducción}_{c}))$$

donde:

* Base: audiencia base según las categorías de los equipos.
* Ponderación: coeficiente del horario asignado.
* Reducción: penalización por coincidencia con otros partidos en el mismo horario.

#Restricciones

Las restricciones se implementan mediante validación o penalización en la función objetivo:

1. Debe haber al menos un partido el viernes a las 20h y otro el lunes a las 20h.
2. Cada partido debe tener asignado exactamente un horario.

En caso de incumplimiento, la solución se penaliza fuertemente en la función objetivo.

#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

El problema consiste en asignar 10 partidos a 10 posibles horarios.

Como los horarios se pueden repetir, cada partido puede tomar cualquiera
de los 10 horarios, por lo que el número total de soluciones es:

|S| = 10^10

Esto supone un espacio de búsqueda muy grande, por lo que no es viable
probar todas las combinaciones posibles.

La evaluación de una solución es rápida (O(n)), ya que solo hay que recorrer
los partidos y calcular la audiencia.

En conjunto, el problema tiene complejidad exponencial, lo que justifica
el uso de algoritmos heurísticos.

#Diseño
- ¿Que técnica utilizo? ¿Por qué?

Se utiliza un algoritmo de búsqueda local (Hill Climbing).

Se parte de una solución inicial aleatoria y se generan soluciones vecinas
cambiando el horario de un partido. Si la nueva solución mejora la audiencia,
se acepta.

Este método es adecuado porque el espacio de búsqueda es muy grande y permite
encontrar buenas soluciones en poco tiempo.

Como desventaja, puede quedarse en óptimos locales y no garantizar la mejor
solución global.

## Código

In [37]:
import random

# 1. Datos del problema

horarios = ['V20', 'S12', 'S16', 'S18', 'S20', 'D12', 'D16', 'D18', 'D20', 'L20']

categorias_audiencia = {
    ('A', 'A'): 2.0, ('A', 'B'): 1.3, ('A', 'C'): 1.0,
    ('B', 'B'): 0.9, ('B', 'C'): 0.75, ('C', 'C'): 0.47
}

ponderacion_horaria = {
    'V20': 0.4, 'L20': 0.4,
    'S12': 0.55, 'S16': 0.7, 'S18': 0.8, 'S20': 1.0,
    'D12': 0.45, 'D16': 0.75, 'D18': 0.85, 'D20': 1.0
}

reduccion_coincidencia = {
    0: 0.0, 1: 0.25, 2: 0.45, 3: 0.60, 4: 0.70,
    5: 0.75, 6: 0.78, 7: 0.80, 8: 0.80
}

# Jornada
jornada = [
    ('B', 'A'), ('B', 'A'), ('C', 'C'), ('B', 'A'), ('C', 'C'),
    ('B', 'C'), ('B', 'B'), ('B', 'B'), ('B', 'C'), ('B', 'B')
]

# 2. Función de audiencia
def calcular_audiencia(asignacion):

    # Contar coincidencias por horario
    cont = {}
    for h in asignacion:
        cont[h] = cont.get(h, 0) + 1

    # Restricción: viernes y lunes obligatorios
    if cont.get('V20', 0) == 0 or cont.get('L20', 0) == 0:
        return 0

    audiencia_total = 0

    for i, h in enumerate(asignacion):
        partido = jornada[i]

        # Obtener base
        cat = tuple(sorted(partido))
        base = categorias_audiencia.get(cat, 0.47)

        # Penalización por coincidencias
        n_coincidencias = cont[h] - 1
        p_red = reduccion_coincidencia.get(n_coincidencias, 0.80)

        audiencia_partido = base * ponderacion_horaria[h] * (1 - p_red)
        audiencia_total += audiencia_partido

    return audiencia_total


# 3. Solución inicial
def solucion_inicial():
    return [random.choice(horarios) for _ in range(10)]


# 4. Vecino
def vecino(sol):
    nueva = sol.copy()
    partido_a_cambiar = random.randint(0, 9)
    nueva[partido_a_cambiar] = random.choice(horarios)
    return nueva


# 5. Hill Climbing
def optimizar_horarios(iteraciones=5000):

    mejor_solucion = solucion_inicial()
    mejor_audiencia = calcular_audiencia(mejor_solucion)

    for _ in range(iteraciones):

        nueva_solucion = vecino(mejor_solucion)
        nueva_audiencia = calcular_audiencia(nueva_solucion)

        if nueva_audiencia > mejor_audiencia:
            mejor_audiencia = nueva_audiencia
            mejor_solucion = nueva_solucion

    return mejor_solucion, mejor_audiencia


# 6. Ejecutar
sol, aud = optimizar_horarios()

print(f"Mejor audiencia encontrada: {aud:.2f} millones\n")

for i, h in enumerate(sol):
    print(f"Partido {i+1} {jornada[i]}: {h}")

Mejor audiencia encontrada: 6.77 millones

Partido 1 ('B', 'A'): D16
Partido 2 ('B', 'A'): S20
Partido 3 ('C', 'C'): L20
Partido 4 ('B', 'A'): D20
Partido 5 ('C', 'C'): S12
Partido 6 ('B', 'C'): V20
Partido 7 ('B', 'B'): S16
Partido 8 ('B', 'B'): S18
Partido 9 ('B', 'C'): D12
Partido 10 ('B', 'B'): D18
